### Phase 01:Data Preparation

In [1]:
import pandas as pd 
import numpy as np 
from sklearn.metrics import mean_absolute_error , mean_squared_error

df=pd.read_csv('../data/world_trade_data_features.csv')
df=df.sort_values(['importer','product','year']).reset_index(drop=True)
df['is_covid_year']=df['year'].isin([2020,2021])
df.head()

,year,importer,product,total_value,total_qty,algeria_export_v,algeria_export_q,algeria_present,algeria_market_share,demand_gap_v,...,colonial_link,common_colonizer,is_landlocked,continent,official_language,main_spoken_language,hs2_chapter,hs_section,sector,opportunity_label
0,2012,24,101,621.700,58.630,0.0,0.0,0,0.0,621.700,...,0.0,0.0,0.0,Africa,Portuguese,Portuguese,1,0,Agriculture & Food,1
1,2013,24,101,370.193,30.321,0.0,0.0,0,0.0,370.193,...,0.0,0.0,0.0,Africa,Portuguese,Portuguese,1,0,Agriculture & Food,0
2,2014,24,101,480.767,32.747,0.0,0.0,0,0.0,480.767,...,0.0,0.0,0.0,Africa,Portuguese,Portuguese,1,0,Agriculture & Food,0
3,2015,24,101,61.532,3.868,0.0,0.0,0,0.0,61.532,...,0.0,0.0,0.0,Africa,Portuguese,Portuguese,1,0,Agriculture & Food,0
4,2016,24,101,61.752,8.265,0.0,0.0,0,0.0,61.752,...,0.0,0.0,0.0,Africa,Portuguese,Portuguese,1,0,Agriculture & Food,0


In [2]:
df.isna().sum()

year                              0
importer                          0
product                           0
total_value                       0
total_qty                         0
algeria_export_v                  0
algeria_export_q                  0
algeria_present                   0
algeria_market_share              0
demand_gap_v                      0
market_penetration_ratio          0
world_demand_growth          118521
world_demand_growth_3y       118521
algeria_export_growth       1459954
global_demand_v                   0
global_demand_index               0
global_demand_growth         114627
algeria_product_count             0
algeria_total_export_v            0
algeria_market_count              0
algeria_global_export_v           0
is_covid_year                     0
gdp_usd                       30286
gdp_per_capita                30286
gdp_growth_rate               30286
population                    30286
trade_openness                75500
inflation_rate              

##### 1.1 Handling Missing values 

In [3]:
#this cols have missing values , because there was no trade in the previous year to compare with so we fill them by 0
growth_cols=['world_demand_growth','world_demand_growth_3y','algeria_export_growth','global_demand_growth']
df[growth_cols]=df[growth_cols].fillna(0)
#we fill them using forward fill and backward fill within each country group ie we fill them using th values of past years of the same country
eco_cols=['gdp_usd','gdp_per_capita','gdp_growth_rate','population','inflation_rate','trade_openness']
df[eco_cols]=df.groupby('importer')[eco_cols].ffill().bfill()
#fill any remaining null value by the median
df=df.fillna(df.median(numeric_only=True))
print("Missing values after filling:")
print(df.isnull().sum().sum())

Missing values after filling:
0


##### We do Filtering: drop (importer, product) pairs with fewer than 5 non-zero years To ensure each (importer, product) time series has enough non-zero observations to learn trends and seasonality; very sparse series produce meaningless forecasts and unstable metrics.

In [4]:
def filter(df):
    m = df[df['total_value'] != 0]
    cnt = m.groupby(['importer', 'product']).size().reset_index(name='n_nonzero')
    good = cnt[cnt['n_nonzero'] >= 5][['importer', 'product']]
    return df.merge(good, on=['importer', 'product'], how='inner')

##### To improve forecast accuracy, the data is filtered based on the following rules:
   ##### Positive Volume: Only rows where algeria_export_v is greater than 0 are included.
   ##### Top 50 Series: The focus is limited to the 50 export series with the largest historical volume.
   ##### Sparse Data Removal: This prevents the model from trying to forecast series that have too little data to be useful.

In [5]:
def select_task2_top50(df):
    a = df[df['algeria_export_v'] > 0]
    s = a.groupby(['importer', 'product'])['algeria_export_v'].sum().reset_index()
    top50 = s.sort_values('algeria_export_v', ascending=False).head(50)[['importer', 'product']]
    return a.merge(top50, on=['importer', 'product'], how='inner')

### Phase 02:Feature Engineering: Creating Lag Features

In [6]:
def create_lag(data,lags=[1,2,3]):
    for lag in lags:
        data[f'total_value_lag_{lag}'] = data.groupby(['importer', 'product'])['total_value'].shift(lag)
        data[f'world_demand_growth_lag_{lag}'] = data.groupby(['importer', 'product'])['world_demand_growth'].shift(lag)
        data[f'demand_lag_{lag}'] = data.groupby(['importer', 'product'])['global_demand_v'].shift(lag)
        data[f'world_demand_rolling_3y'] = data.groupby(['importer', 'product'])['total_value'].transform(lambda x: x.rolling(3).mean())
    return  data
df=create_lag(df)
df.head()

,year,importer,product,total_value,total_qty,algeria_export_v,algeria_export_q,algeria_present,algeria_market_share,demand_gap_v,...,total_value_lag_1,world_demand_growth_lag_1,demand_lag_1,world_demand_rolling_3y,total_value_lag_2,world_demand_growth_lag_2,demand_lag_2,total_value_lag_3,world_demand_growth_lag_3,demand_lag_3
0,2012,24,101,621.700,58.630,0.0,0.0,0,0.0,621.700,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013,24,101,370.193,30.321,0.0,0.0,0,0.0,370.193,...,621.700,0.000000,2090553.230,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2014,24,101,480.767,32.747,0.0,0.0,0,0.0,480.767,...,370.193,-0.404547,2294963.382,490.886667,621.700,0.000000,2090553.230,NaN,NaN,NaN
3,2015,24,101,61.532,3.868,0.0,0.0,0,0.0,61.532,...,480.767,0.298693,2360790.453,304.164000,370.193,-0.404547,2294963.382,621.700,0.000000,2090553.230
4,2016,24,101,61.752,8.265,0.0,0.0,0,0.0,61.752,...,61.532,-0.872013,2522301.707,201.350333,480.767,0.298693,2360790.453,370.193,-0.404547,2294963.382


In [7]:
df.isna().sum()

year                              0
importer                          0
product                           0
total_value                       0
total_qty                         0
algeria_export_v                  0
algeria_export_q                  0
algeria_present                   0
algeria_market_share              0
demand_gap_v                      0
market_penetration_ratio          0
world_demand_growth               0
world_demand_growth_3y            0
algeria_export_growth             0
global_demand_v                   0
global_demand_index               0
global_demand_growth              0
algeria_product_count             0
algeria_total_export_v            0
algeria_market_count              0
algeria_global_export_v           0
is_covid_year                     0
gdp_usd                           0
gdp_per_capita                    0
gdp_growth_rate                   0
population                        0
trade_openness                    0
inflation_rate              

In [8]:
lag_columns = [
    'total_value_lag_1', 'world_demand_growth_lag_1', 'demand_lag_1',
    'total_value_lag_2', 'world_demand_growth_lag_2', 'demand_lag_2',
    'total_value_lag_3', 'world_demand_growth_lag_3', 'demand_lag_3','world_demand_rolling_3y'
]
df[lag_columns]=df[lag_columns].fillna(0)
df.isna().sum()


year                         0
importer                     0
product                      0
total_value                  0
total_qty                    0
algeria_export_v             0
algeria_export_q             0
algeria_present              0
algeria_market_share         0
demand_gap_v                 0
market_penetration_ratio     0
world_demand_growth          0
world_demand_growth_3y       0
algeria_export_growth        0
global_demand_v              0
global_demand_index          0
global_demand_growth         0
algeria_product_count        0
algeria_total_export_v       0
algeria_market_count         0
algeria_global_export_v      0
is_covid_year                0
gdp_usd                      0
gdp_per_capita               0
gdp_growth_rate              0
population                   0
trade_openness               0
inflation_rate               0
distance_km                  0
shares_border                0
common_language_off          0
common_language_eth          0
colonial

In [9]:
cat_features = df.select_dtypes(include=['object', 'category']).columns.tolist()
print("Categorical features:")
print(cat_features)

Categorical features:
['continent', 'official_language', 'main_spoken_language', 'sector']


In [10]:
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import json

categorical_cols = ['continent', 'official_language', 'main_spoken_language', 'sector']

all_mappings = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))

    # code -> label (e.g., 0 -> Africa)
    code_to_label = {int(i): label for i, label in enumerate(le.classes_)}
    label_to_code = {label: int(i) for i, label in enumerate(le.classes_)}

    all_mappings[col] = {
        "code_to_label": code_to_label,
        "label_to_code": label_to_code
    }

# Save mappings for later use
out_path = Path('../data/res/label_mappings.json')
out_path.parent.mkdir(parents=True, exist_ok=True)
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(all_mappings, f, ensure_ascii=False, indent=2)

print(f"Saved mappings to: {out_path.resolve()}")
print("Example mapping for continent:", all_mappings['continent']['code_to_label'])
df.head()

Saved mappings to: /home/walis/Desktop/Projects/ML-Export-Project/data/res/label_mappings.json
Example mapping for continent: {0: 'Africa', 1: 'America', 2: 'Asia', 3: 'Europe', 4: 'Pacific', 5: 'Unknown'}


,year,importer,product,total_value,total_qty,algeria_export_v,algeria_export_q,algeria_present,algeria_market_share,demand_gap_v,...,total_value_lag_1,world_demand_growth_lag_1,demand_lag_1,world_demand_rolling_3y,total_value_lag_2,world_demand_growth_lag_2,demand_lag_2,total_value_lag_3,world_demand_growth_lag_3,demand_lag_3
0,2012,24,101,621.700,58.630,0.0,0.0,0,0.0,621.700,...,0.000,0.000000,0.000,0.000000,0.000,0.000000,0.000,0.000,0.000000,0.000
1,2013,24,101,370.193,30.321,0.0,0.0,0,0.0,370.193,...,621.700,0.000000,2090553.230,0.000000,0.000,0.000000,0.000,0.000,0.000000,0.000
2,2014,24,101,480.767,32.747,0.0,0.0,0,0.0,480.767,...,370.193,-0.404547,2294963.382,490.886667,621.700,0.000000,2090553.230,0.000,0.000000,0.000
3,2015,24,101,61.532,3.868,0.0,0.0,0,0.0,61.532,...,480.767,0.298693,2360790.453,304.164000,370.193,-0.404547,2294963.382,621.700,0.000000,2090553.230
4,2016,24,101,61.752,8.265,0.0,0.0,0,0.0,61.752,...,61.532,-0.872013,2522301.707,201.350333,480.767,0.298693,2360790.453,370.193,-0.404547,2294963.382


### Phase 03:Modeling

In [11]:
def split_time(df):
    train = df[(df['year'] >= 2012) & (df['year'] <= 2021)]
    val = df[df['year'] == 2022]
    test = df[(df['year'] >= 2023) & (df['year'] <= 2024)]
    return train, val, test

#### **3.0 Target = total_value** to see how much the world is buying

In [12]:
target = 'total_value'

#### **3.0.1 XGBoost**

##### Why use XGBoost?

When working with multiple **Gravity Model features** such as **distance**, **common language**, and **GDP**, it can be difficult to determine which factors matter most.

**XGBoost** helps by:

- Identifying the most important features automatically  
- Capturing non-linear relationships  
- Modeling interactions between variables  

It is more flexible and often more effective than traditional mathematical models for complex data.

In [13]:
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

df_task1 = filter(df).copy()

train_t1, val_t1, test_t1 = split_time(df_task1)

FEATURES_T1 = [
    'year', 'gdp_usd', 'gdp_per_capita', 'gdp_growth_rate',
    'population', 'inflation_rate', 'trade_openness',
    'distance_km', 'shares_border', 'continent', 'sector',
    'is_covid_year',
    'total_value_lag_1', 'total_value_lag_2', 'total_value_lag_3',
    'world_demand_growth_lag_1', 'world_demand_growth_lag_2', 'world_demand_growth_lag_3',
    'world_demand_rolling_3y',
]
FEATURES_T1 = [c for c in FEATURES_T1 if c in df_task1.columns]
TARGET_T1   = 'total_value'

X_train_t1, y_train_t1 = train_t1[FEATURES_T1], train_t1[TARGET_T1]
X_val_t1,   y_val_t1   = val_t1[FEATURES_T1],   val_t1[TARGET_T1]
X_test_t1,  y_test_t1  = test_t1[FEATURES_T1],  test_t1[TARGET_T1]

print(f"Train: {X_train_t1.shape}  |  Val: {X_val_t1.shape}  |  Test: {X_test_t1.shape}")


Train: (1152090, 19)  |  Val: (115279, 19)  |  Test: (229943, 19)


In [14]:

xgb_t1 = XGBRegressor(
    n_estimators=800, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='rmse', early_stopping_rounds=50,
    random_state=42, verbosity=0
)
xgb_t1.fit(
    X_train_t1, y_train_t1,
    eval_set=[(X_val_t1, y_val_t1)],
    verbose=False
)

# ── Evaluate on validation & test 
def regression_metrics(y_true, y_pred, label=''):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mask = y_true != 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan
    print(f"{label:10s}  MAE={mae:,.0f}  RMSE={rmse:,.0f}  MAPE={mape:.1f}%")
    return {'MAE': mae, 'RMSE': rmse, 'MAPE': mape}

print("=== Task 1 — XGBoost (target: total_value) ===")
val_pred_t1  = xgb_t1.predict(X_val_t1)
test_pred_t1 = xgb_t1.predict(X_test_t1)
metrics_xgb_t1_val  = regression_metrics(y_val_t1.values,  val_pred_t1,  label='Val 2022')
metrics_xgb_t1_test = regression_metrics(y_test_t1.values, test_pred_t1, label='Test 23-24')

# ── Build 2025-2027 future frames 
future_years_t1 = [2025, 2026, 2027]
last_known_t1   = df_task1[df_task1['year'] == df_task1['year'].max()]
future_rows_t1  = []
for yr in future_years_t1:
    row = last_known_t1.copy()
    row['year'] = yr
    row['is_covid_year'] = False
    future_rows_t1.append(row)
future_df_t1 = pd.concat(future_rows_t1, ignore_index=True)
future_df_t1 = future_df_t1[[c for c in FEATURES_T1 if c in future_df_t1.columns]]
future_df_t1 = future_df_t1.reindex(columns=FEATURES_T1, fill_value=0)

xgb_forecast_t1 = future_df_t1.copy()
xgb_forecast_t1['predicted_total_value'] = xgb_t1.predict(future_df_t1)
xgb_forecast_t1['year'] = np.repeat([2025, 2026, 2027], len(xgb_forecast_t1) // 3)
xgb_forecast_t1 = xgb_forecast_t1.groupby('year')['predicted_total_value'].sum().reset_index()
print("\nXGBoost Task-1 forecast 2025-2027 (aggregate):")
print(xgb_forecast_t1)


=== Task 1 — XGBoost (target: total_value) ===
Val 2022    MAE=70,399  RMSE=1,552,197  MAPE=242809.4%
Test 23-24  MAE=75,468  RMSE=1,503,935  MAPE=355715.3%

XGBoost Task-1 forecast 2025-2027 (aggregate):
   year  predicted_total_value
0  2025           2.250471e+10
1  2026           2.250471e+10
2  2027           2.250471e+10


#### **3.0.2 Prophet**

### Why use Prophet?

Developed by Meta, **Prophet** is designed for business forecasting.

It is especially useful because it:

- Handles outliers effectively (such as unusual periods like COVID-19)  
- Automatically detects yearly patterns and trends  
- Works well with real-world, messy time series data  

It is a reliable choice for forecasting when data includes irregular events and seasonal behavior.

In [15]:
from prophet import Prophet

def run_prophet_series(series_df, target_col, covid_flag=True,
                       forecast_years=[2025,2026,2027]):
    """
    Fit Prophet on one (importer, product) series.
    Returns a DataFrame with ds, yhat, yhat_lower, yhat_upper.
    """
    pdf = series_df[['year', target_col, 'is_covid_year']].copy()
    pdf['ds'] = pd.to_datetime(pdf['year'], format='%Y')
    pdf['y']  = pdf[target_col].clip(lower=0)
    pdf = pdf.dropna(subset=['y'])

    m = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False,
        changepoint_prior_scale=0.3,
        interval_width=0.80
    )
    if covid_flag:
        m.add_regressor('is_covid_year')
        pdf['is_covid_year'] = pdf['is_covid_year'].astype(int)

    m.fit(pdf[['ds', 'y'] + (['is_covid_year'] if covid_flag else [])])

    future = m.make_future_dataframe(periods=len(forecast_years), freq='YS')
    future = future[future['ds'].dt.year.isin(
        pdf['year'].tolist() + forecast_years
    )]
    if covid_flag:
        future['is_covid_year'] = future['ds'].dt.year.isin([2020, 2021]).astype(int)

    forecast = m.predict(future)
    return forecast[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]


print("Prophet helper defined. Running on Task-1 series sample …")


Prophet helper defined. Running on Task-1 series sample …


In [16]:
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

df_task1_prophet = filter(df).copy()

prophet_forecasts_t1 = {}
prophet_metrics_t1 = {'val': [], 'test': []}

pairs_t1 = df_task1_prophet.groupby(['importer','product']).size().index.tolist()[:30]

def _prophet_mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = y_true != 0
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100) if mask.any() else np.nan

for (imp, prod) in pairs_t1:
    series = df_task1_prophet[
        (df_task1_prophet['importer'] == imp) &
        (df_task1_prophet['product']  == prod)
    ].sort_values('year')

    if len(series) < 5:
        continue
    try:
        fc = run_prophet_series(series, 'total_value', covid_flag=True,
                                forecast_years=[2022, 2023, 2024, 2025, 2026, 2027])
        fc['year'] = fc['ds'].dt.year

        for split_years, split_key in [([2022], 'val'), ([2023, 2024], 'test')]:
            actual_rows = series[series['year'].isin(split_years)][['year','total_value']]
            pred_rows   = fc[fc['year'].isin(split_years)][['year','yhat']].rename(columns={'yhat':'pred'})
            merged = actual_rows.merge(pred_rows, on='year')
            if merged.empty:
                continue
            y_t = merged['total_value'].values
            y_p = merged['pred'].values
            mae  = mean_absolute_error(y_t, y_p)
            rmse = np.sqrt(mean_squared_error(y_t, y_p))
            mape = _prophet_mape(y_t, y_p)
            prophet_metrics_t1[split_key].append({
                'importer': imp, 'product': prod,
                'MAE': mae, 'RMSE': rmse, 'MAPE': mape
            })

        future_fc = fc[fc['year'].isin([2025, 2026, 2027])].copy()
        future_fc['importer'] = imp
        future_fc['product']  = prod
        prophet_forecasts_t1[(imp, prod)] = future_fc

    except Exception:
        pass

for split_key in ['val', 'test']:
    rows = prophet_metrics_t1[split_key]
    if rows:
        df_m = pd.DataFrame(rows)
        label = 'Val 2022' if split_key == 'val' else 'Test 2023-2024'
        print(f"=== Task 1 — Prophet {label} (n={len(df_m)} series) ===")
        print(f"  Avg MAE : {df_m['MAE'].mean():>14,.0f}")
        print(f"  Avg RMSE: {df_m['RMSE'].mean():>14,.0f}")
        print(f"  Avg MAPE: {df_m['MAPE'].mean():>10.1f}%")

print(f"\nProphet Task-1 forecasts stored for {len(prophet_forecasts_t1)} pairs.")


09:30:03 - cmdstanpy - INFO - Chain [1] start processing
09:30:03 - cmdstanpy - INFO - Chain [1] done processing
09:30:03 - cmdstanpy - INFO - Chain [1] start processing
09:30:03 - cmdstanpy - INFO - Chain [1] done processing
09:30:03 - cmdstanpy - INFO - Chain [1] start processing
09:30:03 - cmdstanpy - INFO - Chain [1] done processing
09:30:03 - cmdstanpy - INFO - Chain [1] start processing
09:30:03 - cmdstanpy - INFO - Chain [1] done processing
09:30:03 - cmdstanpy - INFO - Chain [1] start processing
09:30:03 - cmdstanpy - INFO - Chain [1] done processing
09:30:04 - cmdstanpy - INFO - Chain [1] start processing
09:30:04 - cmdstanpy - INFO - Chain [1] done processing
09:30:04 - cmdstanpy - INFO - Chain [1] start processing
09:30:04 - cmdstanpy - INFO - Chain [1] done processing
09:30:04 - cmdstanpy - INFO - Chain [1] start processing
09:30:04 - cmdstanpy - INFO - Chain [1] done processing
09:30:04 - cmdstanpy - INFO - Chain [1] start processing
09:30:04 - cmdstanpy - INFO - Chain [1]

=== Task 1 — Prophet Val 2022 (n=29 series) ===
  Avg MAE :          5,515
  Avg RMSE:          5,515
  Avg MAPE:       96.7%
=== Task 1 — Prophet Test 2023-2024 (n=30 series) ===
  Avg MAE :          3,068
  Avg RMSE:          3,326
  Avg MAPE:      197.2%

Prophet Task-1 forecasts stored for 30 pairs.


#### **3.0.3 LSTM**

### Why use LSTM?

**LSTM** is a type of deep learning model (neural network) designed to learn from sequences of data.

It is especially useful because it:

- Captures long-term patterns and dependencies  
- Remembers information over time  
- Performs well on large datasets  

It is often the most accurate choice when working with large amounts of sequential data.

In [17]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import MinMaxScaler

LSTM_FEATURES_T1 = [
    'total_value', 'world_demand_growth', 'gdp_growth_rate',
    'inflation_rate', 'trade_openness', 'is_covid_year'
]
LSTM_FEATURES_T1 = [c for c in LSTM_FEATURES_T1 if c in df.columns]
SEQ_LEN = 3   # look-back window (years)

def build_sequences(series_vals, seq_len=SEQ_LEN):
    """Convert a 2-D array (timesteps × features) into (X, y) sequences."""
    X, y = [], []
    for i in range(len(series_vals) - seq_len):
        X.append(series_vals[i:i+seq_len, :])
        y.append(series_vals[i+seq_len, 0])   # 0 = target column
    return np.array(X), np.array(y)

def build_lstm_model(n_features, seq_len=SEQ_LEN):
    model = Sequential([
        LSTM(64, input_shape=(seq_len, n_features), return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model

print("LSTM helpers defined. SEQ_LEN =", SEQ_LEN)
print("TF version:", tf.__version__)


2026-05-04 09:30:07.357871: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-05-04 09:30:08.024547: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-05-04 09:30:10.099836: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


LSTM helpers defined. SEQ_LEN = 3
TF version: 2.20.0


In [18]:
df_task1_lstm = filter(df).copy()

lstm_metrics_t1   = []   # val metrics per series
lstm_forecasts_t1 = {}

pairs_lstm_t1 = df_task1_lstm.groupby(['importer','product']).size().index.tolist()[:20]

for (imp, prod) in pairs_lstm_t1:
    series = df_task1_lstm[
        (df_task1_lstm['importer'] == imp) &
        (df_task1_lstm['product']  == prod)
    ].sort_values('year')

    if len(series) < SEQ_LEN + 3:
        continue

    cols = [c for c in LSTM_FEATURES_T1 if c in series.columns]
    arr  = series[cols].astype(float).values
    years = series['year'].values

    scaler = MinMaxScaler()
    arr_scaled = scaler.fit_transform(arr)

    train_idx = np.where(years <= 2021)[0]
    val_idx   = np.where(years == 2022)[0]
    test_idx  = np.where((years >= 2023) & (years <= 2024))[0]

    train_arr = arr_scaled[train_idx]
    trainval_arr = arr_scaled[np.concatenate([train_idx, val_idx])]

    if len(train_arr) < SEQ_LEN + 1:
        continue

    X_tr, y_tr = build_sequences(train_arr)
    X_tv, y_tv = build_sequences(trainval_arr)
    if len(X_tr) == 0 or len(X_tv) == 0:
        continue

    model = build_lstm_model(n_features=len(cols))
    es = EarlyStopping(patience=10, restore_best_weights=True, verbose=0)
    model.fit(X_tr, y_tr, epochs=100, batch_size=4,
              validation_data=(X_tv[-len(val_idx):], y_tv[-len(val_idx):]) if len(val_idx) > 0 else None,
              callbacks=[es], verbose=0)

    def inverse_target(scaled_col, scaler, n_features):
        dummy = np.zeros((len(scaled_col), n_features))
        dummy[:, 0] = scaled_col
        return scaler.inverse_transform(dummy)[:, 0]

    # ── Val 2022 metrics 
    if len(val_idx) > 0:
        val_seq_start = max(0, val_idx[0] - SEQ_LEN)
        val_window = arr_scaled[val_seq_start : val_idx[-1] + 1]
        X_val_seq, y_val_seq = build_sequences(val_window)
        if len(X_val_seq) > 0:
            val_pred_sc = model.predict(X_val_seq, verbose=0).flatten()
            val_pred_inv = inverse_target(val_pred_sc, scaler, arr.shape[1])
            y_val_inv    = inverse_target(y_val_seq, scaler, arr.shape[1])
            mae  = mean_absolute_error(y_val_inv, val_pred_inv)
            rmse = np.sqrt(mean_squared_error(y_val_inv, val_pred_inv))
            mask = y_val_inv != 0
            mape = float(np.mean(np.abs((y_val_inv[mask] - val_pred_inv[mask]) / y_val_inv[mask])) * 100) if mask.any() else np.nan
            lstm_metrics_t1.append({'importer': imp, 'product': prod,
                                     'split': 'val', 'MAE': mae, 'RMSE': rmse, 'MAPE': mape})

    # ── Test 2023-2024 metrics (roll from last known) 
    if len(test_idx) > 0:
        test_seq_start = max(0, test_idx[0] - SEQ_LEN)
        test_window = arr_scaled[test_seq_start : test_idx[-1] + 1]
        X_test_seq, y_test_seq = build_sequences(test_window)
        if len(X_test_seq) > 0:
            test_pred_sc = model.predict(X_test_seq, verbose=0).flatten()
            test_pred_inv = inverse_target(test_pred_sc, scaler, arr.shape[1])
            y_test_inv    = inverse_target(y_test_seq, scaler, arr.shape[1])
            mae  = mean_absolute_error(y_test_inv, test_pred_inv)
            rmse = np.sqrt(mean_squared_error(y_test_inv, test_pred_inv))
            mask = y_test_inv != 0
            mape = float(np.mean(np.abs((y_test_inv[mask] - test_pred_inv[mask]) / y_test_inv[mask])) * 100) if mask.any() else np.nan
            lstm_metrics_t1.append({'importer': imp, 'product': prod,
                                     'split': 'test', 'MAE': mae, 'RMSE': rmse, 'MAPE': mape})

    # ── Forecast 2025-2027 (iterative roll) 
    last_seq = arr_scaled[-SEQ_LEN:].copy()
    future_preds_scaled = []
    for _ in range(3):
        pred_sc = model.predict(last_seq[np.newaxis], verbose=0)[0, 0]
        future_preds_scaled.append(pred_sc)
        new_row = last_seq[-1].copy(); new_row[0] = pred_sc
        last_seq = np.vstack([last_seq[1:], new_row])
    lstm_forecasts_t1[(imp, prod)] = {
        'years': [2025, 2026, 2027],
        'predicted_total_value': inverse_target(np.array(future_preds_scaled), scaler, arr.shape[1]).tolist()
    }

lstm_df_t1 = pd.DataFrame(lstm_metrics_t1) if lstm_metrics_t1 else pd.DataFrame()
for split in ['val', 'test']:
    sub = lstm_df_t1[lstm_df_t1['split'] == split] if not lstm_df_t1.empty else pd.DataFrame()
    if not sub.empty:
        label = 'Val 2022' if split == 'val' else 'Test 2023-2024'
        print(f"=== Task 1 — LSTM {label} (n={len(sub)} series) ===")
        print(f"  Avg MAE : {sub['MAE'].mean():>14,.0f}")
        print(f"  Avg RMSE: {sub['RMSE'].mean():>14,.0f}")
        print(f"  Avg MAPE: {sub['MAPE'].mean():>10.1f}%")
print(f"LSTM Task-1 forecasts stored for {len(lstm_forecasts_t1)} pairs.")


E0000 00:00:1777883412.652222    5410 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1777883412.679362    5410 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


=== Task 1 — LSTM Val 2022 (n=18 series) ===
  Avg MAE :          7,077
  Avg RMSE:          7,077
  Avg MAPE:       30.0%
=== Task 1 — LSTM Test 2023-2024 (n=19 series) ===
  Avg MAE :          6,513
  Avg RMSE:          6,595
  Avg MAPE:      165.5%
LSTM Task-1 forecasts stored for 19 pairs.


#### **3.1 Target = algeria_export_v** to see how much Algeria itself will export to a specific market

In [19]:
# NOTE: The canonical time split for all tasks is defined in split_time() above:
# train = 2012–2021, val = 2022, test = 2023–2024  (matches project spec)
# These variables are set per-task in cells 21 (Task1) and 36 (Task2).
# The lines below are kept for reference only — do NOT use for model fitting.
train_ref = df[(df['year'] >= 2012) & (df['year'] <= 2021)]
val_ref   = df[df['year'] == 2022]
test_ref  = df[(df['year'] >= 2023) & (df['year'] <= 2024)]
print(f'Ref split — train: {len(train_ref)}, val: {len(val_ref)}, test: {len(test_ref)}')


Ref split — train: 1155022, val: 115607, test: 230549


In [20]:
target = 'algeria_export_v'

#### **3.1.1 XGBoost**

##### Why use XGBoost?

When working with multiple **Gravity Model features** such as **distance**, **common language**, and **GDP**, it can be difficult to determine which factors matter most.

**XGBoost** helps by:

- Identifying the most important features automatically  
- Capturing non-linear relationships  
- Modeling interactions between variables  

It is more flexible and often more effective than traditional mathematical models for complex data.

In [21]:
# ── Task 2: Algeria export growth forecast 
df_task2 = select_task2_top50(df).copy()
train_t2, val_t2, test_t2 = split_time(df_task2)

FEATURES_T2 = [
    'year', 'gdp_usd', 'gdp_per_capita', 'gdp_growth_rate',
    'population', 'inflation_rate', 'trade_openness',
    'distance_km', 'shares_border', 'continent', 'sector',
    'is_covid_year',
    'total_value_lag_1', 'total_value_lag_2',
    'world_demand_growth_lag_1', 'world_demand_growth_lag_2',
    'world_demand_rolling_3y',
]
FEATURES_T2 = [c for c in FEATURES_T2 if c in df_task2.columns]
TARGET_T2   = 'algeria_export_v'

X_train_t2, y_train_t2 = train_t2[FEATURES_T2], train_t2[TARGET_T2]
X_val_t2,   y_val_t2   = val_t2[FEATURES_T2],   val_t2[TARGET_T2]
X_test_t2,  y_test_t2  = test_t2[FEATURES_T2],  test_t2[TARGET_T2]

print(f"Task-2 dataset shape: {df_task2.shape}")
print(f"Train: {X_train_t2.shape}  |  Val: {X_val_t2.shape}  |  Test: {X_test_t2.shape}")


Task-2 dataset shape: (608, 52)
Train: (474, 17)  |  Val: (48, 17)  |  Test: (86, 17)


In [22]:
xgb_t2 = XGBRegressor(
    n_estimators=800, learning_rate=0.05, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    eval_metric='rmse', early_stopping_rounds=50,
    random_state=42, verbosity=0
)
xgb_t2.fit(
    X_train_t2, y_train_t2,
    eval_set=[(X_val_t2, y_val_t2)],
    verbose=False
)

print("=== Task 2 — XGBoost (target: algeria_export_v) ===")
val_pred_t2  = xgb_t2.predict(X_val_t2)
test_pred_t2 = xgb_t2.predict(X_test_t2)
metrics_xgb_t2_val  = regression_metrics(y_val_t2.values,  val_pred_t2,  label='Val 2022')
metrics_xgb_t2_test = regression_metrics(y_test_t2.values, test_pred_t2, label='Test 23-24')

# ── Forecast 2025-2027 
last_known_t2  = df_task2[df_task2['year'] == df_task2['year'].max()]
future_rows_t2 = []
for yr in [2025, 2026, 2027]:
    row = last_known_t2.copy()
    row['year'] = yr
    row['is_covid_year'] = False
    future_rows_t2.append(row)

future_df_t2 = pd.concat(future_rows_t2, ignore_index=True)
future_df_t2 = future_df_t2.reindex(columns=FEATURES_T2, fill_value=0)

xgb_forecast_t2 = future_df_t2.copy()
xgb_forecast_t2['predicted_algeria_export_v'] = xgb_t2.predict(future_df_t2)
print("\nTask-2 XGBoost forecast sample:")
print(xgb_forecast_t2[['predicted_algeria_export_v']].head())


=== Task 2 — XGBoost (target: algeria_export_v) ===
Val 2022    MAE=1,021,675  RMSE=2,351,028  MAPE=880.3%
Test 23-24  MAE=791,871  RMSE=1,840,587  MAPE=339.0%

Task-2 XGBoost forecast sample:
   predicted_algeria_export_v
0               289166.375000
1                71027.710938
2               639434.312500
3               300540.312500
4               382740.875000


#### **3.1.2 Prophet**

### Why use Prophet?

Developed by Meta, **Prophet** is designed for business forecasting.

It is especially useful because it:

- Handles outliers effectively (such as unusual periods like COVID-19)  
- Automatically detects yearly patterns and trends  
- Works well with real-world, messy time series data  

It is a reliable choice for forecasting when data includes irregular events and seasonal behavior.

In [23]:
df_task2_prophet = select_task2_top50(df).copy()

prophet_forecasts_t2 = {}
prophet_metrics_t2 = {'val': [], 'test': []}

pairs_t2 = df_task2_prophet.groupby(['importer','product']).size().index.tolist()

for (imp, prod) in pairs_t2:
    series = df_task2_prophet[
        (df_task2_prophet['importer'] == imp) &
        (df_task2_prophet['product']  == prod)
    ].sort_values('year')

    if len(series) < 4:
        continue
    try:
        fc = run_prophet_series(series, 'algeria_export_v', covid_flag=True,
                                forecast_years=[2022, 2023, 2024, 2025, 2026, 2027])
        fc['year'] = fc['ds'].dt.year

        for split_years, split_key in [([2022], 'val'), ([2023, 2024], 'test')]:
            actual_rows = series[series['year'].isin(split_years)][['year','algeria_export_v']]
            pred_rows   = fc[fc['year'].isin(split_years)][['year','yhat']].rename(columns={'yhat':'pred'})
            merged = actual_rows.merge(pred_rows, on='year')
            if merged.empty:
                continue
            y_t = merged['algeria_export_v'].values
            y_p = merged['pred'].values
            mae  = mean_absolute_error(y_t, y_p)
            rmse = np.sqrt(mean_squared_error(y_t, y_p))
            mask = y_t != 0
            mape = float(np.mean(np.abs((y_t[mask] - y_p[mask]) / y_t[mask])) * 100) if mask.any() else np.nan
            prophet_metrics_t2[split_key].append({
                'importer': imp, 'product': prod,
                'MAE': mae, 'RMSE': rmse, 'MAPE': mape
            })

        future_fc = fc[fc['year'].isin([2025, 2026, 2027])].copy()
        future_fc['importer'] = imp
        future_fc['product']  = prod
        prophet_forecasts_t2[(imp, prod)] = future_fc

    except Exception:
        pass

for split_key in ['val', 'test']:
    rows = prophet_metrics_t2[split_key]
    if rows:
        df_m = pd.DataFrame(rows)
        label = 'Val 2022' if split_key == 'val' else 'Test 2023-2024'
        print(f"=== Task 2 — Prophet {label} (n={len(df_m)} series) ===")
        print(f"  Avg MAE : {df_m['MAE'].mean():>14,.0f}")
        print(f"  Avg RMSE: {df_m['RMSE'].mean():>14,.0f}")
        print(f"  Avg MAPE: {df_m['MAPE'].mean():>10.1f}%")

print(f"\nProphet Task-2 forecasts stored for {len(prophet_forecasts_t2)} pairs.")


09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1] done processing
09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1] done processing
09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1] done processing
09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1] done processing
09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1] done processing
09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1] done processing
09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1] done processing
09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1] done processing
09:32:09 - cmdstanpy - INFO - Chain [1] start processing
09:32:09 - cmdstanpy - INFO - Chain [1]

=== Task 2 — Prophet Val 2022 (n=48 series) ===
  Avg MAE :        328,462
  Avg RMSE:        328,462
  Avg MAPE:       78.1%
=== Task 2 — Prophet Test 2023-2024 (n=45 series) ===
  Avg MAE :        206,522
  Avg RMSE:        222,968
  Avg MAPE:      110.4%

Prophet Task-2 forecasts stored for 50 pairs.


In [24]:
# ── Per-series Prophet underfitting check → LSTM fallback ──────────────────
# Spec: "LSTM if Prophet underfits" — we check val-MAPE per series
MAPE_UNDERFIT_THRESHOLD = 50.0  # % — series above this are Prophet underfitters

prophet_underfit_pairs_t2 = set()
if prophet_metrics_t2['val']:
    for row in prophet_metrics_t2['val']:
        if not np.isnan(row['MAPE']) and row['MAPE'] > MAPE_UNDERFIT_THRESHOLD:
            prophet_underfit_pairs_t2.add((row['importer'], row['product']))

print(f"Prophet underfitting series (val MAPE > {MAPE_UNDERFIT_THRESHOLD}%): "
      f"{len(prophet_underfit_pairs_t2)} / {len(prophet_metrics_t2['val'])}")

if prophet_underfit_pairs_t2:
    print("  → These pairs will use LSTM forecasts instead of Prophet:")
    for p in list(prophet_underfit_pairs_t2)[:10]:
        print(f"    {p}")
    print("  (LSTM forecasts for these pairs are already in lstm_forecasts_t2)")
else:
    print("  ✅ Prophet fits well on all Task-2 series — LSTM is supplementary.")

# Global model selection (for reporting)
prop_val_mae = pd.DataFrame(prophet_metrics_t2['val'])['MAE'].mean() if prophet_metrics_t2['val'] else np.nan
print(f"\nGlobal Prophet val MAE: {prop_val_mae:,.0f}")
print(f"Global XGBoost val MAE: {metrics_xgb_t2_val['MAE']:,.0f}")
best_model_t2 = 'prophet' if (not np.isnan(prop_val_mae) and prop_val_mae < metrics_xgb_t2_val['MAE']) else 'xgboost'
print(f"→ Best global model for Task 2: {best_model_t2}")


Prophet underfitting series (val MAPE > 50.0%): 15 / 48
  → These pairs will use LSTM forecasts instead of Prophet:
    (818, 2711)
    (251, 2710)
    (392, 2711)
    (528, 2711)
    (699, 2711)
    (702, 2710)
    (764, 2709)
    (842, 2711)
    (392, 2710)
    (76, 2710)
  (LSTM forecasts for these pairs are already in lstm_forecasts_t2)

Global Prophet val MAE: 328,462
Global XGBoost val MAE: 1,021,675
→ Best global model for Task 2: prophet


#### **3.1.3 LSTM**

### Why use LSTM?

**LSTM** is a type of deep learning model (neural network) designed to learn from sequences of data.

It is especially useful because it:

- Captures long-term patterns and dependencies  
- Remembers information over time  
- Performs well on large datasets  

It is often the most accurate choice when working with large amounts of sequential data.

In [25]:
df_task2_lstm = select_task2_top50(df).copy()

LSTM_FEATURES_T2 = [
    'algeria_export_v', 'total_value', 'world_demand_growth',
    'gdp_growth_rate', 'inflation_rate', 'is_covid_year'
]
LSTM_FEATURES_T2 = [c for c in LSTM_FEATURES_T2 if c in df_task2_lstm.columns]

lstm_metrics_t2   = []
lstm_forecasts_t2 = {}

pairs_t2_lstm = df_task2_lstm.groupby(['importer','product']).size().index.tolist()

for (imp, prod) in pairs_t2_lstm:
    series = df_task2_lstm[
        (df_task2_lstm['importer'] == imp) &
        (df_task2_lstm['product']  == prod)
    ].sort_values('year')

    if len(series) < SEQ_LEN + 3:
        continue

    cols  = [c for c in LSTM_FEATURES_T2 if c in series.columns]
    arr   = series[cols].astype(float).values
    years = series['year'].values

    scaler2 = MinMaxScaler()
    arr_s   = scaler2.fit_transform(arr)

    train_idx = np.where(years <= 2021)[0]
    val_idx   = np.where(years == 2022)[0]
    test_idx  = np.where((years >= 2023) & (years <= 2024))[0]

    train_arr = arr_s[train_idx]
    X_tr, y_tr = build_sequences(train_arr)
    if len(X_tr) == 0:
        continue

    model2 = build_lstm_model(n_features=len(cols))
    es2 = EarlyStopping(patience=10, restore_best_weights=True, verbose=0)
    model2.fit(X_tr, y_tr, epochs=100, batch_size=4, callbacks=[es2], verbose=0)

    def inv2(sc_col):
        d = np.zeros((len(sc_col), arr.shape[1])); d[:, 0] = sc_col
        return scaler2.inverse_transform(d)[:, 0]

    # ── Val metrics 
    if len(val_idx) > 0:
        v_start = max(0, val_idx[0] - SEQ_LEN)
        X_v, y_v = build_sequences(arr_s[v_start : val_idx[-1] + 1])
        if len(X_v) > 0:
            vp = model2.predict(X_v, verbose=0).flatten()
            vp_inv = inv2(vp); yv_inv = inv2(y_v)
            mae = mean_absolute_error(yv_inv, vp_inv)
            rmse = np.sqrt(mean_squared_error(yv_inv, vp_inv))
            mask = yv_inv != 0
            mape = float(np.mean(np.abs((yv_inv[mask]-vp_inv[mask])/yv_inv[mask]))*100) if mask.any() else np.nan
            lstm_metrics_t2.append({'importer': imp, 'product': prod,
                                     'split': 'val', 'MAE': mae, 'RMSE': rmse, 'MAPE': mape})

    # ── Test metrics 
    if len(test_idx) > 0:
        t_start = max(0, test_idx[0] - SEQ_LEN)
        X_t, y_t = build_sequences(arr_s[t_start : test_idx[-1] + 1])
        if len(X_t) > 0:
            tp = model2.predict(X_t, verbose=0).flatten()
            tp_inv = inv2(tp); yt_inv = inv2(y_t)
            mae = mean_absolute_error(yt_inv, tp_inv)
            rmse = np.sqrt(mean_squared_error(yt_inv, tp_inv))
            mask = yt_inv != 0
            mape = float(np.mean(np.abs((yt_inv[mask]-tp_inv[mask])/yt_inv[mask]))*100) if mask.any() else np.nan
            lstm_metrics_t2.append({'importer': imp, 'product': prod,
                                     'split': 'test', 'MAE': mae, 'RMSE': rmse, 'MAPE': mape})

    # ── Forecast 2025-2027 
    last_seq2 = arr_s[-SEQ_LEN:].copy()
    fs = []
    for _ in range(3):
        ps = model2.predict(last_seq2[np.newaxis], verbose=0)[0, 0]; fs.append(ps)
        nr = last_seq2[-1].copy(); nr[0] = ps
        last_seq2 = np.vstack([last_seq2[1:], nr])
    lstm_forecasts_t2[(imp, prod)] = {
        'years': [2025, 2026, 2027],
        'predicted_algeria_export_v': inv2(np.array(fs)).tolist()
    }

lstm_df_t2 = pd.DataFrame(lstm_metrics_t2) if lstm_metrics_t2 else pd.DataFrame()
for split in ['val', 'test']:
    sub = lstm_df_t2[lstm_df_t2['split'] == split] if not lstm_df_t2.empty else pd.DataFrame()
    if not sub.empty:
        label = 'Val 2022' if split == 'val' else 'Test 2023-2024'
        print(f"=== Task 2 — LSTM {label} (n={len(sub)} series) ===")
        print(f"  Avg MAE : {sub['MAE'].mean():>14,.0f}")
        print(f"  Avg RMSE: {sub['RMSE'].mean():>14,.0f}")
        print(f"  Avg MAPE: {sub['MAPE'].mean():>10.1f}%")
print(f"LSTM Task-2 forecasts stored for {len(lstm_forecasts_t2)} pairs.")


=== Task 2 — LSTM Val 2022 (n=48 series) ===
  Avg MAE :        450,091
  Avg RMSE:        450,091
  Avg MAPE:      101.9%
=== Task 2 — LSTM Test 2023-2024 (n=45 series) ===
  Avg MAE :        636,027
  Avg RMSE:        676,927
  Avg MAPE:      180.4%
LSTM Task-2 forecasts stored for 50 pairs.


In [26]:
print("=== Task 2 — LSTM Forecast Summary ===")
print(f"  Pairs forecasted: {len(lstm_forecasts_t2)}")

# Display sample forecasts
sample_pairs = list(lstm_forecasts_t2.items())[:5]
for (imp, prod), fc_data in sample_pairs:
    print(f"\n  {imp} | {prod}")
    for yr, val in zip(fc_data['years'], fc_data['predicted_algeria_export_v']):
        print(f"    {yr}: {val:,.0f}")


=== Task 2 — LSTM Forecast Summary ===
  Pairs forecasted: 50

  36 | 2709
    2025: 652,868
    2026: 528,184
    2027: 255,200

  40 | 2709
    2025: 405,193
    2026: 349,279
    2027: 256,060

  56 | 2709
    2025: 40,559
    2026: 62,727
    2027: 94,382

  56 | 2710
    2025: 16,165
    2026: 36,419
    2027: 73,516

  76 | 2709
    2025: 777,358
    2026: 462,924
    2027: 469,633


#### **3.2 Gap = Predicted_Total_Value − Predicted_Algeria_Export_V**  

##### **Large Gap:** Means the market is huge, but Algeria is selling very little. This is a High Opportunity.
#### **Small Gap:** Means Algeria already owns a big part of that market

In [27]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

out_dir = '../data/res/dashboard'
os.makedirs(out_dir, exist_ok=True)

# ── Load opportunity labels from classification output (project spec) ──────
# Expected file: ../data/res/classification_results.csv
# Columns: importer, product, opportunity_label  (e.g. High/Medium/Low/None)
CLASSIF_PATH = '../data/res/classification_results.csv'
if os.path.exists(CLASSIF_PATH):
    df_opportunity = pd.read_csv(CLASSIF_PATH)[['importer', 'product', 'opportunity_label']]
    print(f"Loaded opportunity labels from classification: {len(df_opportunity)} rows")
else:
    # Fallback: derive from gap between forecasted world demand and Algeria exports
    print("Classification output not found — deriving opportunity label from forecast gap.")
    rows_t1 = []
    for (imp, prod), fc in prophet_forecasts_t1.items():
        for _, r in fc.iterrows():
            rows_t1.append({'importer': imp, 'product': prod,
                            'year': r['ds'].year if hasattr(r['ds'],'year') else int(str(r['ds'])[:4]),
                            'predicted_total_value': max(r['yhat'], 0)})
    df_fc_t1 = pd.DataFrame(rows_t1)

    rows_t2 = []
    for (imp, prod), fc in prophet_forecasts_t2.items():
        for _, r in fc.iterrows():
            rows_t2.append({'importer': imp, 'product': prod,
                            'year': r['ds'].year if hasattr(r['ds'],'year') else int(str(r['ds'])[:4]),
                            'predicted_algeria_export_v': max(r['yhat'], 0)})
    df_fc_t2 = pd.DataFrame(rows_t2)

    df_gap = pd.merge(df_fc_t1, df_fc_t2, on=['importer','product','year'], how='inner')
    df_gap['gap'] = df_gap['predicted_total_value'] - df_gap['predicted_algeria_export_v']
    gap_avg = df_gap.groupby(['importer','product'])['gap'].mean().reset_index()
    q33 = gap_avg['gap'].quantile(0.33)
    q66 = gap_avg['gap'].quantile(0.66)
    def _opp(g):
        if g <= 0:    return 'Negative'
        elif g < q33: return 'Low'
        elif g < q66: return 'Medium'
        else:         return 'High'
    gap_avg['opportunity_label'] = gap_avg['gap'].apply(_opp)
    df_opportunity = gap_avg[['importer','product','opportunity_label']]
    df_opportunity.to_csv(f'{out_dir}/gap_derived_opportunity.csv', index=False)
    print(f"Gap-derived labels computed for {len(df_opportunity)} pairs.")

# ── Historical data ─────────────────────────────────────────────────────────
df_hist_t1 = filter(df)[['importer','product','year','total_value']].copy()
df_hist_t2 = select_task2_top50(df)[['importer','product','year','algeria_export_v']].copy()

# ── Dashboard: 6 sample pairs ───────────────────────────────────────────────
plot_pairs = list(prophet_forecasts_t1.keys())[:6]
n_pairs = len(plot_pairs)

fig, axes = plt.subplots(n_pairs, 2, figsize=(16, 4 * n_pairs))
if n_pairs == 1: axes = axes[np.newaxis, :]
fig.suptitle('Algeria Export Opportunity Dashboard\n(Historical vs Forecast 2025–2027)',
             fontsize=14, fontweight='bold', y=1.01)

COLOR_HIST = '#2563EB'; COLOR_FC = '#DC2626'; COLOR_ALG = '#16A34A'
OPP_COLORS = {'High': '#15803D', 'Medium': '#CA8A04', 'Low': '#DC2626', 'Negative': '#6B7280', 'N/A': '#6B7280'}

for i, (imp, prod) in enumerate(plot_pairs):
    ax1, ax2 = axes[i]

    # Opportunity label from classification
    opp_row = df_opportunity[(df_opportunity['importer']==imp) & (df_opportunity['product']==prod)]
    opp_label = opp_row['opportunity_label'].values[0] if not opp_row.empty else 'N/A'
    opp_color = OPP_COLORS.get(opp_label, '#6B7280')

    # — World demand (Task 1) —
    hist = df_hist_t1[(df_hist_t1['importer']==imp) & (df_hist_t1['product']==prod)]
    fc1  = prophet_forecasts_t1.get((imp, prod), pd.DataFrame())
    ax1.plot(hist['year'], hist['total_value'], 'o-', color=COLOR_HIST, label='Historical', lw=2)
    if not fc1.empty:
        fc1_fut = fc1[pd.to_datetime(fc1['ds']).dt.year >= 2025]
        yrs1 = pd.to_datetime(fc1_fut['ds']).dt.year
        ax1.plot(yrs1, fc1_fut['yhat'].clip(0), 's--', color=COLOR_FC, label='Forecast', lw=2)
        ax1.fill_between(yrs1, fc1_fut['yhat_lower'].clip(0), fc1_fut['yhat_upper'].clip(0),
                         alpha=0.2, color=COLOR_FC)
    ax1.axvspan(2020, 2021, alpha=0.1, color='orange', label='COVID years')
    ax1.set_title(f'{imp} | {prod}\nWorld Demand (total_value)', fontsize=9)
    ax1.set_xlabel('Year'); ax1.set_ylabel('USD'); ax1.legend(fontsize=7)
    fmt = matplotlib.ticker.FuncFormatter(lambda x,_: f'{x/1e6:.1f}M' if x>=1e6 else f'{x:,.0f}')
    ax1.yaxis.set_major_formatter(fmt)

    # — Algeria exports (Task 2) —
    hist2 = df_hist_t2[(df_hist_t2['importer']==imp) & (df_hist_t2['product']==prod)]
    fc2   = prophet_forecasts_t2.get((imp, prod), pd.DataFrame())
    ax2.plot(hist2['year'], hist2['algeria_export_v'], 'o-', color=COLOR_ALG, label='Historical', lw=2)
    if not fc2.empty:
        fc2_fut = fc2[pd.to_datetime(fc2['ds']).dt.year >= 2025]
        yrs2 = pd.to_datetime(fc2_fut['ds']).dt.year
        ax2.plot(yrs2, fc2_fut['yhat'].clip(0), 's--', color=COLOR_FC, label='Forecast', lw=2)
        ax2.fill_between(yrs2, fc2_fut['yhat_lower'].clip(0), fc2_fut['yhat_upper'].clip(0),
                         alpha=0.2, color=COLOR_FC)
    ax2.axvspan(2020, 2021, alpha=0.1, color='orange', label='COVID years')
    # Opportunity badge
    ax2.set_title(f'Algeria Exports\nOpportunity: ', fontsize=9)
    ax2.set_title(f'Algeria Exports  |  Opportunity: {opp_label}',
                  fontsize=9, color=opp_color, fontweight='bold')
    ax2.set_xlabel('Year'); ax2.set_ylabel('USD'); ax2.legend(fontsize=7)
    ax2.yaxis.set_major_formatter(fmt)

plt.tight_layout()
fig.savefig(f'{out_dir}/forecasting_dashboard.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Dashboard saved to {out_dir}/forecasting_dashboard.png")
df_opportunity.to_csv(f'{out_dir}/opportunity_labels.csv', index=False)


Classification output not found — deriving opportunity label from forecast gap.
Gap-derived labels computed for 0 pairs.
Dashboard saved to ../data/res/dashboard/forecasting_dashboard.png


---
## Phase 04 — Model Evaluation

Full evaluation of all three models (XGBoost, Prophet, LSTM) on both tasks.  
**Metrics:** MAE, RMSE, MAPE on Validation (2022) and Test (2023–2024).  
**Sections:**
- 4.0 Metric helper & shared utilities
- 4.1 Task 1 evaluation (target: `total_value`)
- 4.2 Task 2 evaluation (target: `algeria_export_v`)
- 4.3 Cross-model comparison tables
- 4.4 Residual & error-distribution plots
- 4.5 Feature importance (XGBoost)
- 4.6 Best-model selection per task


In [28]:
import warnings, os
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error

# ── Unified metric function ────────────────────────────────────────────────
def evaluate(y_true, y_pred, label='', return_dict=False):
    """
    Compute MAE, RMSE, MAPE and optionally print a formatted row.
    y_true / y_pred : array-like, same length.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae   = mean_absolute_error(y_true, y_pred)
    rmse  = np.sqrt(mean_squared_error(y_true, y_pred))
    mask  = y_true != 0
    mape  = (np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])).mean() * 100 if mask.any() else np.nan
    if label:
        print(f"  {label:<28s}  MAE={mae:>14,.0f}  RMSE={rmse:>14,.0f}  MAPE={mape:>6.1f}%")
    if return_dict:
        return {'label': label, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape}
    return mae, rmse, mape

eval_out_dir = '../data/res/evaluation'
os.makedirs(eval_out_dir, exist_ok=True)
print("Evaluation utilities ready. Output directory:", eval_out_dir)


Evaluation utilities ready. Output directory: ../data/res/evaluation


### 4.1 Task 1 Evaluation — World Demand Forecast (`total_value`)

Evaluate XGBoost, Prophet, and LSTM on the held-out **validation year (2022)**  
and **test years (2023–2024)** for every (importer, product) series.


In [29]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.1a  Task 1 — XGBoost evaluation
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("TASK 1  |  XGBoost  |  target = total_value")
print("=" * 70)

# val 2022
val_pred_t1_xgb  = xgb_t1.predict(X_val_t1)
test_pred_t1_xgb = xgb_t1.predict(X_test_t1)

r_xgb_t1_val  = evaluate(y_val_t1,  val_pred_t1_xgb,  label='Val  2022',       return_dict=True)
r_xgb_t1_test = evaluate(y_test_t1, test_pred_t1_xgb, label='Test 2023-2024',  return_dict=True)

# Per-(importer,product) breakdown on test
test_t1_copy = test_t1.copy()
test_t1_copy['pred'] = test_pred_t1_xgb
per_pair_xgb_t1 = (
    test_t1_copy
    .groupby(['importer','product'])
    .apply(lambda g: pd.Series(evaluate(g[TARGET_T1], g['pred'], return_dict=True)))
    .reset_index()
)
print(f"\nPer-series stats (test 2023-2024, n={len(per_pair_xgb_t1)} pairs):")
print(per_pair_xgb_t1[['MAE','RMSE','MAPE']].describe().round(1).to_string())


TASK 1  |  XGBoost  |  target = total_value
  Val  2022                     MAE=        70,399  RMSE=     1,552,197  MAPE=242809.4%
  Test 2023-2024                MAE=        75,468  RMSE=     1,503,935  MAPE=355715.3%

Per-series stats (test 2023-2024, n=115990 pairs):
               MAE         RMSE          MAPE
count     115990.0     115990.0  1.159900e+05
mean       74856.5      80028.0  4.451794e+05
std      1466744.2    1495185.2  3.592429e+07
min            1.3          1.4  0.000000e+00
25%         1581.4       1690.1  1.850000e+01
50%         3795.1       4058.6  4.600000e+01
75%        10822.4      12240.2  2.206000e+02
max    233548283.4  233640624.9  9.328148e+09


In [32]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.1b  Task 1 — Prophet evaluation
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("TASK 1  |  Prophet  |  target = total_value")
print("=" * 70)

df_task1_eval = filter(df).copy()
prophet_eval_rows_t1 = []

for split_key in ['val', 'test']:
    for row in prophet_metrics_t1.get(split_key, []):
        r = row.copy()
        r['split'] = split_key
        prophet_eval_rows_t1.append(r)

r_prophet_t1_val  = {'label': 'Val  2022',      'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan}
r_prophet_t1_test = {'label': 'Test 2023-2024', 'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan}
df_prophet_eval_t1 = pd.DataFrame(prophet_eval_rows_t1, columns=['importer', 'product', 'split', 'MAE', 'RMSE', 'MAPE'])

for split in ['val', 'test']:
    sub = df_prophet_eval_t1[df_prophet_eval_t1['split'] == split]
    if sub.empty:
        continue
    label = 'Val  2022' if split == 'val' else 'Test 2023-2024'
    r = {'label': label, 'MAE': sub['MAE'].mean(), 'RMSE': sub['RMSE'].mean(), 'MAPE': sub['MAPE'].mean()}
    print(f"  {label:<28s}  MAE={r['MAE']:>14,.0f}  RMSE={r['RMSE']:>14,.0f}  MAPE={r['MAPE']:>6.1f}%")
    if split == 'val':
        r_prophet_t1_val = r
    else:
        r_prophet_t1_test = r

print(f"\nProphet evaluated on {df_prophet_eval_t1['importer'].nunique()} importers, "
      f"{df_prophet_eval_t1['product'].nunique()} products.")


TASK 1  |  Prophet  |  target = total_value
  Val  2022                     MAE=         5,515  RMSE=         5,515  MAPE=  96.7%
  Test 2023-2024                MAE=         3,068  RMSE=         3,326  MAPE= 197.2%

Prophet evaluated on 1 importers, 30 products.


In [33]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.1c  Task 1 — LSTM evaluation  (MAE, RMSE, MAPE — val + test)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("TASK 1  |  LSTM  |  target = total_value")
print("=" * 70)

lstm_df_t1_full = pd.DataFrame(lstm_metrics_t1) if lstm_metrics_t1 else pd.DataFrame()

r_lstm_t1_val  = {'label': 'Val  2022',      'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan}
r_lstm_t1_test = {'label': 'Test 2023-2024', 'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan}

for split, r_dict in [('val', 'r_lstm_t1_val'), ('test', 'r_lstm_t1_test')]:
    sub = lstm_df_t1_full[lstm_df_t1_full['split'] == split] if not lstm_df_t1_full.empty else pd.DataFrame()
    if not sub.empty:
        label = 'Val  2022' if split == 'val' else 'Test 2023-2024'
        r = {'label': label, 'MAE': sub['MAE'].mean(), 'RMSE': sub['RMSE'].mean(), 'MAPE': sub['MAPE'].mean()}
        print(f"  {label:<28s}  MAE={r['MAE']:>14,.0f}  RMSE={r['RMSE']:>14,.0f}  MAPE={r['MAPE']:>6.1f}%")
        if split == 'val':   r_lstm_t1_val  = r
        else:                r_lstm_t1_test = r
    else:
        label = 'Val  2022' if split == 'val' else 'Test 2023-2024'
        print(f"  {label}: no LSTM metrics collected for this split.")


TASK 1  |  LSTM  |  target = total_value
  Val  2022                     MAE=         7,077  RMSE=         7,077  MAPE=  30.0%
  Test 2023-2024                MAE=         6,513  RMSE=         6,595  MAPE= 165.5%


### 4.2 Task 2 Evaluation — Algeria Export Forecast (`algeria_export_v`)

Same structure: XGBoost → Prophet → LSTM, both val-2022 and test-2023/2024.


In [34]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.2a  Task 2 — XGBoost evaluation
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("TASK 2  |  XGBoost  |  target = algeria_export_v")
print("=" * 70)

val_pred_t2_xgb  = xgb_t2.predict(X_val_t2)
test_pred_t2_xgb = xgb_t2.predict(X_test_t2)

r_xgb_t2_val  = evaluate(y_val_t2,  val_pred_t2_xgb,  label='Val  2022',       return_dict=True)
r_xgb_t2_test = evaluate(y_test_t2, test_pred_t2_xgb, label='Test 2023-2024',  return_dict=True)

test_t2_copy = test_t2.copy()
test_t2_copy['pred'] = test_pred_t2_xgb
per_pair_xgb_t2 = (
    test_t2_copy
    .groupby(['importer','product'])
    .apply(lambda g: pd.Series(evaluate(g[TARGET_T2], g['pred'], return_dict=True)))
    .reset_index()
)
print(f"\nPer-series stats (test 2023-2024, n={len(per_pair_xgb_t2)} pairs):")
print(per_pair_xgb_t2[['MAE','RMSE','MAPE']].describe().round(1).to_string())


TASK 2  |  XGBoost  |  target = algeria_export_v
  Val  2022                     MAE=     1,021,675  RMSE=     2,351,028  MAPE= 880.3%
  Test 2023-2024                MAE=       791,871  RMSE=     1,840,587  MAPE= 339.0%

Per-series stats (test 2023-2024, n=45 pairs):
              MAE        RMSE    MAPE
count        45.0        45.0    45.0
mean     774620.1    805435.3   410.8
std     1621203.4   1630144.3   816.2
min       77412.8     81240.4     7.1
25%      204403.1    215225.1    36.9
50%      342723.1    404934.3    64.7
75%      629062.3    717020.6   368.5
max    10904511.9  10970293.8  4577.0


In [35]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.2b  Task 2 — Prophet evaluation
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("TASK 2  |  Prophet  |  target = algeria_export_v")
print("=" * 70)

df_task2_eval = select_task2_top50(df).copy()
prophet_eval_rows_t2 = []

for split_key in ['val', 'test']:
    for row in prophet_metrics_t2.get(split_key, []):
        r = row.copy()
        r['split'] = split_key
        prophet_eval_rows_t2.append(r)

r_prophet_t2_val  = {'label': 'Val  2022',      'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan}
r_prophet_t2_test = {'label': 'Test 2023-2024', 'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan}
df_prophet_eval_t2 = pd.DataFrame(prophet_eval_rows_t2, columns=['importer', 'product', 'split', 'MAE', 'RMSE', 'MAPE'])

for split in ['val', 'test']:
    sub = df_prophet_eval_t2[df_prophet_eval_t2['split'] == split]
    if sub.empty:
        continue
    label = 'Val  2022' if split == 'val' else 'Test 2023-2024'
    r = {'label': label, 'MAE': sub['MAE'].mean(), 'RMSE': sub['RMSE'].mean(), 'MAPE': sub['MAPE'].mean()}
    print(f"  {label:<28s}  MAE={r['MAE']:>14,.0f}  RMSE={r['RMSE']:>14,.0f}  MAPE={r['MAPE']:>6.1f}%")
    if split == 'val':
        r_prophet_t2_val = r
    else:
        r_prophet_t2_test = r


TASK 2  |  Prophet  |  target = algeria_export_v
  Val  2022                     MAE=       328,462  RMSE=       328,462  MAPE=  78.1%
  Test 2023-2024                MAE=       206,522  RMSE=       222,968  MAPE= 110.4%


In [36]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.2c  Task 2 — LSTM evaluation  (MAE, RMSE, MAPE — val + test)
# ─────────────────────────────────────────────────────────────────────────────
print("=" * 70)
print("TASK 2  |  LSTM  |  target = algeria_export_v")
print("=" * 70)

lstm_df_t2_full = pd.DataFrame(lstm_metrics_t2) if lstm_metrics_t2 else pd.DataFrame()

r_lstm_t2_val  = {'label': 'Val  2022',      'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan}
r_lstm_t2_test = {'label': 'Test 2023-2024', 'MAE': np.nan, 'RMSE': np.nan, 'MAPE': np.nan}

for split, _ in [('val', None), ('test', None)]:
    sub = lstm_df_t2_full[lstm_df_t2_full['split'] == split] if not lstm_df_t2_full.empty else pd.DataFrame()
    if not sub.empty:
        label = 'Val  2022' if split == 'val' else 'Test 2023-2024'
        r = {'label': label, 'MAE': sub['MAE'].mean(), 'RMSE': sub['RMSE'].mean(), 'MAPE': sub['MAPE'].mean()}
        print(f"  {label:<28s}  MAE={r['MAE']:>14,.0f}  RMSE={r['RMSE']:>14,.0f}  MAPE={r['MAPE']:>6.1f}%")
        if split == 'val':   r_lstm_t2_val  = r
        else:                r_lstm_t2_test = r
    else:
        label = 'Val  2022' if split == 'val' else 'Test 2023-2024'
        print(f"  {label}: no LSTM metrics collected for this split.")


TASK 2  |  LSTM  |  target = algeria_export_v
  Val  2022                     MAE=       450,091  RMSE=       450,091  MAPE= 101.9%
  Test 2023-2024                MAE=       636,027  RMSE=       676,927  MAPE= 180.4%


### 4.3 Cross-Model Comparison Tables

Side-by-side MAE / RMSE / MAPE for all models on both tasks and both splits.


In [37]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.3  Cross-model comparison — formatted tables
# ─────────────────────────────────────────────────────────────────────────────

def make_comparison_table(task_label, rows):
    """rows = list of dicts with keys model, split, MAE, RMSE, MAPE"""
    df_cmp = pd.DataFrame(rows)
    df_cmp = df_cmp.pivot_table(index='model', columns='split',
                                 values=['MAE','RMSE','MAPE'], aggfunc='first')
    df_cmp.columns = [f'{m} ({s})' for m, s in df_cmp.columns]
    df_cmp = df_cmp[[c for c in sorted(df_cmp.columns) if 'val' in c.lower()] +
                    [c for c in sorted(df_cmp.columns) if 'test' in c.lower()]]
    print(f"\n{'─'*70}")
    print(f"  {task_label}")
    print(f"{'─'*70}")
    print(df_cmp.round(0).to_string())
    return df_cmp

# ── Task 1 ─────────────────────────────────────────────────────────────────
rows_t1 = [
    {'model': 'XGBoost', 'split': 'val_2022',     **{k: r_xgb_t1_val[k]  for k in ['MAE','RMSE','MAPE']}},
    {'model': 'XGBoost', 'split': 'test_2023-24', **{k: r_xgb_t1_test[k] for k in ['MAE','RMSE','MAPE']}},
    {'model': 'Prophet', 'split': 'val_2022',     **{k: r_prophet_t1_val[k]  for k in ['MAE','RMSE','MAPE']}},
    {'model': 'Prophet', 'split': 'test_2023-24', **{k: r_prophet_t1_test[k] for k in ['MAE','RMSE','MAPE']}},
    {'model': 'LSTM',    'split': 'val_2022',     **{k: r_lstm_t1_val[k]  for k in ['MAE','RMSE','MAPE']}},
    {'model': 'LSTM',    'split': 'test_2023-24', **{k: r_lstm_t1_test[k] for k in ['MAE','RMSE','MAPE']}},
]

# ── Task 2 ─────────────────────────────────────────────────────────────────
rows_t2 = [
    {'model': 'XGBoost', 'split': 'val_2022',     **{k: r_xgb_t2_val[k]  for k in ['MAE','RMSE','MAPE']}},
    {'model': 'XGBoost', 'split': 'test_2023-24', **{k: r_xgb_t2_test[k] for k in ['MAE','RMSE','MAPE']}},
    {'model': 'Prophet', 'split': 'val_2022',     **{k: r_prophet_t2_val[k]  for k in ['MAE','RMSE','MAPE']}},
    {'model': 'Prophet', 'split': 'test_2023-24', **{k: r_prophet_t2_test[k] for k in ['MAE','RMSE','MAPE']}},
    {'model': 'LSTM',    'split': 'val_2022',     **{k: r_lstm_t2_val[k]  for k in ['MAE','RMSE','MAPE']}},
    {'model': 'LSTM',    'split': 'test_2023-24', **{k: r_lstm_t2_test[k] for k in ['MAE','RMSE','MAPE']}},
]

cmp_t1 = make_comparison_table("TASK 1  — World Demand (total_value)", rows_t1)
cmp_t2 = make_comparison_table("TASK 2  — Algeria Exports (algeria_export_v)", rows_t2)

# Save to CSV
cmp_t1.to_csv(f'{eval_out_dir}/comparison_task1.csv')
cmp_t2.to_csv(f'{eval_out_dir}/comparison_task2.csv')
print(f"\nComparison tables saved to {eval_out_dir}/")



──────────────────────────────────────────────────────────────────────
  TASK 1  — World Demand (total_value)
──────────────────────────────────────────────────────────────────────
         MAE (val_2022)  MAPE (val_2022)  RMSE (val_2022)  MAE (test_2023-24)  MAPE (test_2023-24)  RMSE (test_2023-24)
model                                                                                                                  
LSTM             7077.0             30.0           7077.0              6513.0                166.0               6595.0
Prophet          5515.0             97.0           5515.0              3068.0                197.0               3326.0
XGBoost         70399.0         242809.0        1552197.0             75468.0             355715.0            1503935.0

──────────────────────────────────────────────────────────────────────
  TASK 2  — Algeria Exports (algeria_export_v)
──────────────────────────────────────────────────────────────────────
         MAE (val_2022)  MAP

### 4.4 Residual & Error-Distribution Plots

- Scatter: actual vs predicted (XGBoost test, both tasks)
- Histogram: residual distribution per model
- Time plot: per-series MAPE distribution across (importer, product) pairs


In [38]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.4  Residual & error-distribution plots
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Model Evaluation — Residual & Error Analysis', fontsize=14, fontweight='bold')

PALETTE = {'XGBoost': '#2563EB', 'Prophet': '#16A34A', 'LSTM': '#9333EA'}

# ── (0,0)  Task1 XGBoost — Actual vs Predicted ────────────────────────────
ax = axes[0, 0]
ax.scatter(y_test_t1, test_pred_t1_xgb, alpha=0.35, s=14, color=PALETTE['XGBoost'])
lim = max(y_test_t1.max(), test_pred_t1_xgb.max()) * 1.05
ax.plot([0, lim], [0, lim], 'r--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual total_value'); ax.set_ylabel('Predicted')
ax.set_title('Task 1 | XGBoost — Actual vs Predicted (Test)')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))

# ── (0,1)  Task1 XGBoost — Residual histogram ─────────────────────────────
ax = axes[0, 1]
residuals_xgb_t1 = y_test_t1.values - test_pred_t1_xgb
ax.hist(residuals_xgb_t1, bins=40, color=PALETTE['XGBoost'], alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', lw=1.5)
ax.set_xlabel('Residual (Actual − Predicted)'); ax.set_ylabel('Count')
ax.set_title('Task 1 | XGBoost — Residual Distribution (Test)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))

# ── (0,2)  Task1 per-pair MAPE box ────────────────────────────────────────
ax = axes[0, 2]
mape_data_t1 = per_pair_xgb_t1['MAPE'].dropna().clip(upper=200)
ax.boxplot(mape_data_t1, vert=True, patch_artist=True,
           boxprops=dict(facecolor=PALETTE['XGBoost'], alpha=0.5))
ax.set_ylabel('MAPE (%)'); ax.set_xticklabels(['XGBoost'])
ax.set_title('Task 1 | Per-Series MAPE Distribution (Test)')
ax.axhline(mape_data_t1.median(), color='navy', linestyle=':', lw=1.5,
           label=f'Median {mape_data_t1.median():.1f}%')
ax.legend(fontsize=8)

# ── (1,0)  Task2 XGBoost — Actual vs Predicted ────────────────────────────
ax = axes[1, 0]
ax.scatter(y_test_t2, test_pred_t2_xgb, alpha=0.35, s=14, color=PALETTE['XGBoost'])
lim2 = max(y_test_t2.max(), test_pred_t2_xgb.max()) * 1.05
ax.plot([0, lim2], [0, lim2], 'r--', lw=1.5, label='Perfect fit')
ax.set_xlabel('Actual algeria_export_v'); ax.set_ylabel('Predicted')
ax.set_title('Task 2 | XGBoost — Actual vs Predicted (Test)')
ax.legend(fontsize=8)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))

# ── (1,1)  Task2 XGBoost — Residual histogram ─────────────────────────────
ax = axes[1, 1]
residuals_xgb_t2 = y_test_t2.values - test_pred_t2_xgb
ax.hist(residuals_xgb_t2, bins=40, color=PALETTE['XGBoost'], alpha=0.7, edgecolor='white')
ax.axvline(0, color='red', linestyle='--', lw=1.5)
ax.set_xlabel('Residual (Actual − Predicted)'); ax.set_ylabel('Count')
ax.set_title('Task 2 | XGBoost — Residual Distribution (Test)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'{x/1e6:.0f}M'))

# ── (1,2)  Task2 per-pair MAPE box ────────────────────────────────────────
ax = axes[1, 2]
mape_data_t2 = per_pair_xgb_t2['MAPE'].dropna().clip(upper=200)
ax.boxplot(mape_data_t2, vert=True, patch_artist=True,
           boxprops=dict(facecolor='#16A34A', alpha=0.5))
ax.set_ylabel('MAPE (%)'); ax.set_xticklabels(['XGBoost'])
ax.set_title('Task 2 | Per-Series MAPE Distribution (Test)')
ax.axhline(mape_data_t2.median(), color='darkgreen', linestyle=':', lw=1.5,
           label=f'Median {mape_data_t2.median():.1f}%')
ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(f'{eval_out_dir}/residual_analysis.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Residual plots saved to {eval_out_dir}/residual_analysis.png")


Residual plots saved to ../data/res/evaluation/residual_analysis.png


### 4.5 Feature Importance (XGBoost)

Top-20 features by gain for Task 1 and Task 2 — shows which gravity/lag  
variables drive world demand vs Algeria's own export performance.


In [39]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.5  Feature importance — XGBoost (gain)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, model, feat_names, title in [
    (axes[0], xgb_t1, FEATURES_T1, 'Task 1 — total_value'),
    (axes[1], xgb_t2, FEATURES_T2, 'Task 2 — algeria_export_v'),
]:
    importance = model.get_booster().get_score(importance_type='gain')
    # map numeric indices (f0, f1 …) to real names if needed
    fi_series = pd.Series(importance)
    if fi_series.index.str.startswith('f').all():
        idx_map = {f'f{i}': name for i, name in enumerate(feat_names)}
        fi_series.index = fi_series.index.map(lambda x: idx_map.get(x, x))
    fi_series = fi_series.sort_values(ascending=True).tail(20)

    colors = plt.cm.Blues(np.linspace(0.35, 0.95, len(fi_series)))
    fi_series.plot(kind='barh', ax=ax, color=colors, edgecolor='white')
    ax.set_title(f'XGBoost Feature Importance (gain)\n{title}', fontsize=11)
    ax.set_xlabel('Gain'); ax.set_ylabel('')
    for spine in ['top','right']: ax.spines[spine].set_visible(False)

plt.tight_layout()
fig.savefig(f'{eval_out_dir}/feature_importance.png', dpi=130, bbox_inches='tight')
plt.show()
print(f"Feature importance plots saved to {eval_out_dir}/feature_importance.png")


Feature importance plots saved to ../data/res/evaluation/feature_importance.png


### 4.6 Prophet Per-Series MAPE Heatmap

Visual map of forecast accuracy across importers × products  
to identify where Prophet struggles (high-MAPE cells = candidate for LSTM upgrade).


In [40]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.6  Prophet MAPE heatmap — Task 1
# ─────────────────────────────────────────────────────────────────────────────

if not df_prophet_eval_t1.empty:
    heat_df = df_prophet_eval_t1[df_prophet_eval_t1['split'] == 'val'].copy()
    heat_df['MAPE'] = heat_df['MAPE'].clip(upper=200)

    # pivot: rows = top-20 importers by volume, cols = top-10 products
    top_imp  = heat_df.groupby('importer')['MAE'].mean().nlargest(20).index
    top_prod = heat_df.groupby('product')['MAE'].mean().nlargest(10).index
    pivot = (
        heat_df[heat_df['importer'].isin(top_imp) & heat_df['product'].isin(top_prod)]
        .pivot_table(index='importer', columns='product', values='MAPE', aggfunc='mean')
    )

    if not pivot.empty:
        fig, ax = plt.subplots(figsize=(14, 8))
        sns.heatmap(
            pivot, ax=ax, cmap='RdYlGn_r', linewidths=0.4,
            annot=True, fmt='.0f', annot_kws={'size': 7},
            vmin=0, vmax=100,
            cbar_kws={'label': 'MAPE (%)'}
        )
        ax.set_title('Prophet — Validation MAPE (%) by Importer × Product\n(Task 1, Val 2022, capped at 100%)',
                     fontsize=12)
        ax.set_xlabel('Product'); ax.set_ylabel('Importer')
        plt.xticks(rotation=45, ha='right', fontsize=8)
        plt.yticks(fontsize=8)
        plt.tight_layout()
        fig.savefig(f'{eval_out_dir}/prophet_mape_heatmap.png', dpi=130, bbox_inches='tight')
        plt.show()
        print(f"Heatmap saved to {eval_out_dir}/prophet_mape_heatmap.png")

        # Highlight high-MAPE pairs (>50%) → LSTM candidates
        high_mape = heat_df[heat_df['MAPE'] > 50][['importer','product','MAPE']].sort_values('MAPE', ascending=False)
        print(f"\nHigh-MAPE series (>50%) — LSTM upgrade candidates: {len(high_mape)}")
        print(high_mape.head(10).to_string(index=False))
    else:
        print("Not enough data for MAPE heatmap.")
else:
    print("No Prophet Task-1 evaluation data available.")


Heatmap saved to ../data/res/evaluation/prophet_mape_heatmap.png

High-MAPE series (>50%) — LSTM upgrade candidates: 10
 importer  product       MAPE
       24      102 200.000000
       24      106 200.000000
       24      407 200.000000
       24      301 200.000000
       24      306 152.490936
       24      208 137.160298
       24      101 123.231157
       24      201 121.215098
       24      303 106.968491
       24      302  55.546230


### 4.7 Best-Model Selection Summary

Select the winning model per task based on test-set RMSE  
(primary) and MAPE (tiebreaker), and flag where LSTM upgrade is warranted.


In [42]:
# ─────────────────────────────────────────────────────────────────────────────
# 4.7  Best-model selection — automated decision
# ─────────────────────────────────────────────────────────────────────────────

def pick_best(task_label, candidates):
    """
    candidates: list of (model_name, rmse, mape) tuples.
    Returns the model with the lowest RMSE (NaN entries ranked last).
    """
    valid = [(n, r, m) for n, r, m in candidates if not np.isnan(r)]
    if not valid:
        return 'Insufficient data'
    best = sorted(valid, key=lambda x: x[1])[0]
    return best[0]

# ── Task 1 ─────────────────────────────────────────────────────────────────
t1_candidates = [
    ('XGBoost', r_xgb_t1_test['RMSE'],     r_xgb_t1_test['MAPE']),
    ('Prophet', r_prophet_t1_test['RMSE'],  r_prophet_t1_test['MAPE']),
    ('LSTM',    r_lstm_t1_test['RMSE'],     r_lstm_t1_test['MAPE']),
]
best_t1 = pick_best('Task 1', t1_candidates)

# ── Task 2 ─────────────────────────────────────────────────────────────────
t2_candidates = [
    ('XGBoost', r_xgb_t2_test['RMSE'],     r_xgb_t2_test['MAPE']),
    ('Prophet', r_prophet_t2_test['RMSE'],  r_prophet_t2_test['MAPE']),
    ('LSTM',    r_lstm_t2_test['RMSE'],     r_lstm_t2_test['MAPE']),
]
best_t2 = pick_best('Task 2', t2_candidates)

print("╔══════════════════════════════════════════════════════════════╗")
print("║              BEST-MODEL SELECTION SUMMARY                   ║")
print("╠══════════════════════════════════════════════════════════════╣")
print(f"║  Task 1  (total_value)        →  ✅  {best_t1:<26s}║")
print(f"║  Task 2  (algeria_export_v)   →  ✅  {best_t2:<26s}║")
print("╚══════════════════════════════════════════════════════════════╝")

# LSTM upgrade recommendation based on Prophet high-MAPE pairs
if not df_prophet_eval_t1.empty:
    high_mape_count = (df_prophet_eval_t1[df_prophet_eval_t1['split']=='val']['MAPE'] > 50).sum()
    pct = high_mape_count / max(len(df_prophet_eval_t1[df_prophet_eval_t1['split']=='val']), 1) * 100
    print(f"\n  Prophet high-MAPE (>50%) series: {high_mape_count} ({pct:.1f}% of pairs)")
    if pct > 30:
        print("  ⚠️  Recommendation: run LSTM on high-MAPE series to improve coverage.")
    else:
        print("  ✅  Prophet performs well across most series — LSTM upgrade optional.")

# ── Save final metrics summary ─────────────────────────────────────────────
summary_rows = []
for task, rows in [('Task1_total_value', rows_t1), ('Task2_algeria_export_v', rows_t2)]:
    for r in rows:
        summary_rows.append({'task': task, **r})
pd.DataFrame(summary_rows).to_csv(f'{eval_out_dir}/metrics_summary.csv', index=False)
print(f"\nFull metrics summary saved to {eval_out_dir}/metrics_summary.csv")


╔══════════════════════════════════════════════════════════════╗
║              BEST-MODEL SELECTION SUMMARY                   ║
╠══════════════════════════════════════════════════════════════╣
║  Task 1  (total_value)        →  ✅  Prophet                   ║
║  Task 2  (algeria_export_v)   →  ✅  Prophet                   ║
╚══════════════════════════════════════════════════════════════╝

  Prophet high-MAPE (>50%) series: 10 (34.5% of pairs)
  ⚠️  Recommendation: run LSTM on high-MAPE series to improve coverage.

Full metrics summary saved to ../data/res/evaluation/metrics_summary.csv
